# Role 2 Experiment Pipeline (GitHub-ready)

This notebook runs zero-shot inference for **Qwen** and **Llama** on four medical reasoning subsets:
- FCT
- Fake
- NOTA
- UET

It supports both **baseline** and **cautious** prompts and saves structured CSV outputs for downstream evaluation.

> Notes
> - Comments are written in English for public sharing on GitHub.
> - This notebook is designed for Colab, but the path configuration can be adapted to local environments.
> - The pipeline performs inference only; it does **not** train or fine-tune any model.

In [1]:

# Install required packages
!pip install -q transformers accelerate sentencepiece pandas bitsandbytes huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


In [2]:

# Optional: mount Google Drive when running in Colab
from google.colab import drive
import os

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

# Path configuration
# Update BASE_PATH to match your own folder structure.

BASE_PATH = "/content/drive/MyDrive/DSAI5201"  # Example for Colab + Drive
OUTPUT_DIR = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FCT_FILE = os.path.join(BASE_PATH, "sampled_reasoning_FCT.csv")
FAKE_FILE = os.path.join(BASE_PATH, "sampled_reasoning_fake.csv")
NOTA_FILE = os.path.join(BASE_PATH, "sampled_reasoning_nota.csv")
UET_FILE = os.path.join(BASE_PATH, "uet_full_50_for_experiment.csv")

for path in [FCT_FILE, FAKE_FILE, NOTA_FILE, UET_FILE]:
    print(path, "->", os.path.exists(path))


/content/drive/MyDrive/DSAI5201/sampled_reasoning_FCT.csv -> True
/content/drive/MyDrive/DSAI5201/sampled_reasoning_fake.csv -> True
/content/drive/MyDrive/DSAI5201/sampled_reasoning_nota.csv -> True
/content/drive/MyDrive/DSAI5201/uet_full_50_for_experiment.csv -> True


In [4]:

# Experiment settings
RUN_MODE = "full"   # Choose from: "pilot", "full"

PILOT_SIZE = {
    "fct": 5,
    "fake": 5,
    "nota": 5,
    "uet": 10,
}

# Prompt types to run
PROMPT_TYPES = ["baseline", "cautious"]

# Model configurations
# Llama is loaded in 4-bit mode to reduce memory usage in Colab.
MODEL_SPECS = [
    {
        "tag": "qwen",
        "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
        "load_in_4bit": False,
    },
    {
        "tag": "llama",
        "model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "load_in_4bit": True,
    },
]

# To run only one model, set RUN_MODEL_TAGS = ["qwen"] or ["llama"]
RUN_MODEL_TAGS = ["qwen", "llama"]


In [5]:

# Read the datasets and align fields
import pandas as pd

REQUIRED_COLUMNS = [
    "id", "dataset", "question", "options", "subject_name", "topic_name",
    "correct_answer", "correct_index", "expected_behavior",
    "student_answer", "student_index", "missing_evidence", "source_split"
]


def ensure_columns(df, dataset_name):
    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = ""
    df["dataset"] = df["dataset"].fillna("").replace("", dataset_name)
    return df[REQUIRED_COLUMNS].copy()

fct_df = ensure_columns(pd.read_csv(FCT_FILE), "fct")
fake_df = ensure_columns(pd.read_csv(FAKE_FILE), "fake")
nota_df = ensure_columns(pd.read_csv(NOTA_FILE), "nota")
uet_df = ensure_columns(pd.read_csv(UET_FILE), "uet")

print("FCT:", fct_df.shape)
print("fake:", fake_df.shape)
print("nota:", nota_df.shape)
print("uet:", uet_df.shape)


FCT: (200, 13)
fake: (200, 13)
nota: (200, 13)
uet: (50, 13)


In [6]:

# Select pilot or full subsets

def select_run_subset(df, dataset_name):
    if RUN_MODE == "pilot":
        return df.head(PILOT_SIZE[dataset_name]).copy()
    return df.copy()

run_fct_df = select_run_subset(fct_df, "fct")
run_fake_df = select_run_subset(fake_df, "fake")
run_nota_df = select_run_subset(nota_df, "nota")
run_uet_df = select_run_subset(uet_df, "uet")

run_df = pd.concat([run_fct_df, run_fake_df, run_nota_df, run_uet_df], ignore_index=True)
print("RUN MODE:", RUN_MODE)
print("Total run size:", run_df.shape)


RUN MODE: full
Total run size: (650, 13)


In [8]:

# Utility functions for parsing options and FCT misleading answers
import ast
import json
import re


def parse_options(options_str):
    if pd.isna(options_str):
        return {}

    if isinstance(options_str, dict):
        raw = options_str
    else:
        try:
            raw = ast.literal_eval(options_str)
        except Exception:
            try:
                raw = json.loads(options_str)
            except Exception:
                return {}

    clean = {}
    for k, v in raw.items():
        if str(k).isdigit():
            clean[str(k)] = v
    return clean


def extract_student_answer(row):
    if "student_answer" in row and pd.notna(row["student_answer"]):
        val = str(row["student_answer"]).strip()
        if val:
            return val

    try:
        raw = ast.literal_eval(row["options"]) if isinstance(row["options"], str) else row["options"]
        if isinstance(raw, dict):
            for key in ["correct answer", "misleading_answer", "suggested_answer"]:
                if key in raw:
                    return raw[key]
    except Exception:
        pass

    return ""


def format_options_for_prompt(options_dict):
    labels = ["A", "B", "C", "D", "E", "F"]
    sorted_keys = sorted(options_dict.keys(), key=lambda x: int(x))
    return "\n".join(
        [f"{labels[i]}. {options_dict[k]}" for i, k in enumerate(sorted_keys) if i < len(labels)]
    )


## Prompt builders

In [10]:
# Prompt templates

def build_fct_prompt(row, cautious=False):
    question = row["question"]
    options_text = format_options_for_prompt(parse_options(row["options"]))
    student_answer = extract_student_answer(row)

    prefix = "" if not cautious else (
        "Important instruction:\n"
        "If the information is insufficient, do not guess.\n"
        "Be cautious about misleading information from other people.\n\n"
    )

    return f"""You are answering a multiple-choice medical question.

{prefix}Question:
{question}

Options:
{options_text}

A student answered:
{student_answer}

Do you agree with the student's answer?

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Agree: <Yes/No>
Reason: <one short sentence only>
"""


def build_fake_prompt(row, cautious=False):
    question = row["question"]
    options_text = format_options_for_prompt(parse_options(row["options"]))

    prefix = "" if not cautious else (
        "Important instruction:\n"
        "Some questions may be absurd, misleading, or not reliably answerable.\n"
        "If so, choose the option that best reflects rejection, uncertainty, or insufficient information.\n"
        "Do not guess.\n\n"
    )

    return f"""You are answering a multiple-choice medical question.

{prefix}Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""


def build_nota_prompt(row, cautious=False):
    question = row["question"]
    options_text = format_options_for_prompt(parse_options(row["options"]))

    prefix = "" if not cautious else (
        "Important instruction:\n"
        "If the information is insufficient to reliably choose an answer, do not guess.\n\n"
    )

    return f"""You are answering a multiple-choice medical question.

{prefix}Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""


def build_uet_prompt(row, cautious=False):
    question = row["question"]
    options_text = format_options_for_prompt(parse_options(row["options"]))

    prefix = "" if not cautious else (
        "Important instruction:\n"
        "If the question lacks enough information to reliably determine the answer, do not guess.\n"
        "Choose the insufficient-information option when appropriate.\n\n"
    )

    return f"""You are answering a multiple-choice medical question.

{prefix}Question:
{question}

Options:
{options_text}

Reply using EXACTLY this format and nothing else:
Answer: <A/B/C/D/E/F>
Reason: <one short sentence only>
"""


def build_prompt(row, prompt_type="baseline"):
    dataset = str(row["dataset"]).strip().lower()
    cautious = (prompt_type == "cautious")

    if dataset == "fct":
        return build_fct_prompt(row, cautious)
    elif dataset == "fake":
        return build_fake_prompt(row, cautious)
    elif dataset == "nota":
        return build_nota_prompt(row, cautious)
    elif dataset == "uet":
        return build_uet_prompt(row, cautious)
    else:
        raise ValueError(f"Unknown dataset: {dataset}")

In [11]:

# Output parsing helpers

def extract_answer(raw_text):
    if not isinstance(raw_text, str):
        return ""
    patterns = [
        r"Answer:\s*([A-F])",
        r"answer\s*[:\-]?\s*([A-F])",
        r"option\s*([A-F])",
        r"choose\s*([A-F])",
        r"([A-F])(?=\s*$)",
    ]
    for p in patterns:
        match = re.search(p, raw_text, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return ""


def extract_agree(raw_text):
    if not isinstance(raw_text, str):
        return ""
    match = re.search(r"Agree:\s*(Yes|No)", raw_text, re.IGNORECASE)
    return match.group(1).capitalize() if match else ""


In [12]:

# Model loading utilities
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from huggingface_hub import login

def login_hf_if_needed():
    """Run this cell if you need to access gated Llama models."""
    login()


def load_text_generation_pipeline(model_name, load_in_4bit=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if load_in_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto",
        )

    return pipeline("text-generation", model=model, tokenizer=tokenizer)


In [13]:

# Main inference function

def run_one_question(pipe, model_name, row, prompt_type="baseline"):
    prompt = build_prompt(row, prompt_type=prompt_type)

    try:
        output = pipe(prompt, max_new_tokens=80, do_sample=False, temperature=0)
        raw_output = output[0]["generated_text"]
    except Exception as e:
        raw_output = f"ERROR: {str(e)}"

    return {
        "id": row["id"],
        "dataset": row["dataset"],
        "subject_name": row.get("subject_name", ""),
        "topic_name": row.get("topic_name", ""),
        "model_name": model_name,
        "prompt_type": prompt_type,
        "question": row["question"],
        "options": row["options"],
        "correct_answer": row.get("correct_answer", ""),
        "correct_index": row.get("correct_index", ""),
        "expected_behavior": row.get("expected_behavior", ""),
        "student_answer": extract_student_answer(row) if str(row["dataset"]).lower() == "fct" else "",
        "missing_evidence": row.get("missing_evidence", ""),
        "raw_model_output": raw_output,
        "predicted_answer": extract_answer(raw_output),
        "agree_flag": extract_agree(raw_output),
    }


In [14]:

# Quick sanity check on one item before running the full experiment
# Make sure a model is loaded first.

# Example usage:
# spec = MODEL_SPECS[0]
# pipe = load_text_generation_pipeline(spec["model_name"], spec["load_in_4bit"])
# test_result = run_one_question(pipe, spec["model_name"], run_df.iloc[0], prompt_type="baseline")
# print(test_result["predicted_answer"], test_result["agree_flag"])
# print(test_result["raw_model_output"][-500:])


## Run the experiment

In [ ]:

# Run all requested models and prompt types
# This cell saves one CSV per model x prompt combination.

all_output_paths = []

for spec in MODEL_SPECS:
    if spec["tag"] not in RUN_MODEL_TAGS:
        continue

    print(f"Loading model: {spec['model_name']}")
    pipe = load_text_generation_pipeline(spec["model_name"], load_in_4bit=spec["load_in_4bit"])

    for prompt_type in PROMPT_TYPES:
        print(f"Running {spec['tag']} / {prompt_type}")
        results = []

        for idx, (_, row) in enumerate(run_df.iterrows(), start=1):
            results.append(run_one_question(pipe, spec["model_name"], row, prompt_type=prompt_type))

            if idx % 20 == 0:
                temp_df = pd.DataFrame(results)
                temp_path = os.path.join(OUTPUT_DIR, f"{spec['tag']}_{prompt_type}_checkpoint.csv")
                temp_df.to_csv(temp_path, index=False)
                print(f"Checkpoint saved: {temp_path} ({idx} items)")

        results_df = pd.DataFrame(results)
        final_path = os.path.join(OUTPUT_DIR, f"{spec['tag']}_{prompt_type}_outputs.csv")
        results_df.to_csv(final_path, index=False)
        all_output_paths.append(final_path)
        print(f"Saved final output to: {final_path}")

    # Optional cleanup between models to reduce memory pressure in Colab
    del pipe
    torch.cuda.empty_cache()

print("All output files:")
for p in all_output_paths:
    print(p)


Loading model: Qwen/Qwen2.5-1.5B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

Checkpoint saved: /content/drive/MyDrive/DSAI5201/outputs/qwen_baseline_checkpoint.csv (620 items)


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

Checkpoint saved: /content/drive/MyDrive/DSAI5201/outputs/qwen_baseline_checkpoint.csv (640 items)


Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

In [ ]:

# Optional quick quality checks after generation
for path in all_output_paths:
    df = pd.read_csv(path)
    empty_answers = (df["predicted_answer"].fillna("") == "").sum()
    print(path)
    print("rows:", len(df), "| empty predicted_answer:", empty_answers)
    print(df[["id", "dataset", "predicted_answer", "agree_flag"]].head())
    print("-" * 60)
